# Feedforward Neural Network for MNIST Digit Classification

## Overview
This notebook implements a simple Feedforward Neural Network (FFNN) to classify handwritten digits from the MNIST dataset. 

**Dataset:** MNIST (Modified National Institute of Standards and Technology) contains 70,000 grayscale images (28×28 pixels) of handwritten digits (0-9).
- **Training samples:** 60,000
- **Test samples:** 10,000
- **Classes:** 10 (digits 0-9)
- **Input features:** 784 (28×28 flattened pixels)

**Architecture:** 3-layer FFNN with ReLU activations, Dropout regularization, and softmax output.

**Goal:** Achieve >97% accuracy on the test set.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns
import warnings
import os
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Configure device (CPU or GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [ ]:
# Define data transformation: normalize to [0,1]
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL image to tensor and scales to [0,1]
])

# Load MNIST dataset
print("Loading MNIST dataset...")

# Set root directory for dataset (handles Kaggle offline vs local/online)
data_root = './data'
if os.path.exists('/kaggle/input'):
    # Search for MNIST folder in /kaggle/input
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'MNIST' in dirs or 'raw' in dirs:
            data_root = root
            break

print(f"Using dataset root: {data_root}")
try:
    train_dataset = datasets.MNIST(root=data_root, train=True, transform=transform, download=True)
    test_dataset = datasets.MNIST(root=data_root, train=False, transform=transform, download=True)
except Exception as e:
    print(f"Online download failed/not possible: {e}")
    print("Attempting to load from local directory without download...")
    train_dataset = datasets.MNIST(root=data_root, train=True, transform=transform, download=False)
    test_dataset = datasets.MNIST(root=data_root, train=False, transform=transform, download=False)

# Create train/validation split (80/20)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])

# Create DataLoaders with optimizations for Kaggle
batch_size = 64
num_workers = 2 if torch.cuda.is_available() else 0
pin_memory = torch.cuda.is_available()

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Batch size: {batch_size}")

Loading MNIST dataset...


100.0%
100.0%
100.0%
100.0%

Training samples: 48000
Validation samples: 12000
Test samples: 10000
Batch size: 64


In [ ]:
# Explore the dataset
print("Exploring MNIST dataset...")
sample_batch, sample_labels = next(iter(train_loader))
print(f"Sample batch shape: {sample_batch.shape}")  # (batch_size, 1, 28, 28)
print(f"Sample labels shape: {sample_labels.shape}")  # (batch_size,)
print(f"Pixel value range: [{sample_batch.min():.4f}, {sample_batch.max():.4f}]")

# Count class distribution
class_counts = np.bincount(train_dataset.dataset.targets[train_dataset.indices].numpy())
print("\nClass distribution in training set:")
for digit, count in enumerate(class_counts):
    print(f"  Digit {digit}: {count} samples")

In [ ]:
# Display sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample MNIST Images', fontsize=14, fontweight='bold')

sample_batch, sample_labels = next(iter(train_loader))
for idx in range(10):
    ax = axes[idx // 5, idx % 5]
    img = sample_batch[idx].squeeze().numpy()
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Label: {sample_labels[idx].item()}', fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("Sample images displayed successfully.")

In [ ]:
# Data Preprocessing: Normalize and Flatten
print("Preprocessing data: Normalization and Flattening")

# MNIST statistics for standardization
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081

# Create preprocessing function
def preprocess_batch(batch):
    """Flatten and normalize batch of images"""
    batch = batch.view(batch.size(0), -1)  # Flatten: (batch_size, 1, 28, 28) -> (batch_size, 784)
    batch = (batch - MNIST_MEAN) / MNIST_STD  # Standardize
    return batch

# Test preprocessing
sample_batch, _ = next(iter(train_loader))
processed = preprocess_batch(sample_batch)
print(f"Original batch shape: {sample_batch.shape}")
print(f"Processed batch shape: {processed.shape}")
print(f"Processed mean: {processed.mean():.6f}, Processed std: {processed.std():.6f}")
print("Preprocessing verified successfully.")

## Model Architecture

**Feedforward Neural Network (FFNN) Design:**
- **Input Layer:** 784 neurons (28×28 flattened pixels)
- **Hidden Layer 1:** 128 neurons + ReLU activation + Dropout(0.2)
- **Hidden Layer 2:** 64 neurons + ReLU activation + Dropout(0.2)
- **Hidden Layer 3:** 32 neurons + ReLU activation + Dropout(0.2)
- **Output Layer:** 10 neurons (one per digit 0-9) + Softmax (via CrossEntropyLoss)

**Rationale:**
- **Progressive dimensionality reduction:** 784 → 128 → 64 → 32 → 10 prevents overfitting
- **ReLU activation:** Fast, avoids vanishing gradient problem
- **Dropout (0.2):** Randomly deactivates 20% of neurons during training; improves generalization
- **CrossEntropyLoss:** Combines LogSoftmax and NLLLoss; best for multi-class classification

**Expected performance:** >97% test accuracy

In [ ]:
# Define FFNN Model
class FFNN(nn.Module):
    """Feedforward Neural Network for MNIST Classification"""
    def __init__(self, input_size=784, hidden1=128, hidden2=64, hidden3=32, num_classes=10, dropout_rate=0.2):
        super(FFNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1)
        self.bn1 = nn.BatchNorm1d(hidden1)
        self.dropout1 = nn.Dropout(dropout_rate)
        
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.bn2 = nn.BatchNorm1d(hidden2)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        self.fc3 = nn.Linear(hidden2, hidden3)
        self.bn3 = nn.BatchNorm1d(hidden3)
        self.dropout3 = nn.Dropout(dropout_rate)
        
        self.fc4 = nn.Linear(hidden3, num_classes)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        # Flatten input: (batch_size, 1, 28, 28) -> (batch_size, 784)
        x = x.view(x.size(0), -1)
        
        # Layer 1
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout1(x)
        
        # Layer 2
        x = self.fc2(x)
        x = self.bn2(x)
        x = self.relu(x)
        x = self.dropout2(x)
        
        # Layer 3
        x = self.fc3(x)
        x = self.bn3(x)
        x = self.relu(x)
        x = self.dropout3(x)
        
        # Output layer (no activation; handled by CrossEntropyLoss)
        x = self.fc4(x)
        return x

# Initialize model
model = FFNN().to(device)
print("Model created successfully.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Loss Function & Optimizer Setup
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training configuration
num_epochs = 15
patience = 3  # Early stopping patience
best_val_loss = float('inf')
epochs_without_improvement = 0

print("Training configuration:")
print(f"  Loss function: CrossEntropyLoss")
print(f"  Optimizer: Adam (lr=0.001)")
print(f"  Epochs: {num_epochs}")
print(f"  Early stopping patience: {patience}")
print(f"  Device: {device}")

## Training Methodology

**Training Loop:**
1. **Forward Pass:** Feed batch through model, get predictions
2. **Compute Loss:** Compare predictions with true labels using CrossEntropyLoss
3. **Backward Pass:** Compute gradients via backpropagation
4. **Optimizer Step:** Update model weights using Adam optimizer
5. **Accumulate Metrics:** Track loss and accuracy

**Validation Loop:**
- Evaluate model on validation set without gradient computation (no_grad())
- Compute validation loss and accuracy
- Use best validation loss for model checkpoint

**Early Stopping:**
- If validation loss doesn't improve for 3 consecutive epochs, stop training
- Save best model weights based on validation performance

In [ ]:
# Training Function: Train for one epoch
def train_epoch(model, train_loader, loss_fn, optimizer, device):
    """Train the model for one epoch"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    # Wrap with tqdm progress bar
    progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc="Training", leave=False)
    for batch_idx, (images, labels) in progress_bar:
        images, labels = images.to(device), labels.to(device)
        
        # Preprocess: normalize and flatten
        images = (images - MNIST_MEAN) / MNIST_STD
        
        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        # Update progress bar
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy

# Validation Function: Validate on validation set
def validate_epoch(model, val_loader, loss_fn, device):
    """Validate the model on validation set"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    # Wrap with tqdm progress bar
    progress_bar = tqdm(val_loader, desc="Validating", leave=False)
    with torch.no_grad():
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            # Preprocess: normalize and flatten
            images = (images - MNIST_MEAN) / MNIST_STD
            
            # Forward pass
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            # Track metrics
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100.0 * correct / total
    return avg_loss, accuracy

print("Training and validation functions defined successfully.")

In [ ]:
# Main Training Loop
print("Starting training...")
print("-" * 70)

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []
best_model_state = None

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, loss_fn, optimizer, device)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    
    # Validate
    val_loss, val_acc = validate_epoch(model, val_loader, loss_fn, device)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)
    
    print(f"Epoch {epoch+1:2d}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        best_model_state = model.state_dict().copy()
        print(f"  ✓ Best model updated (Val Loss: {best_val_loss:.4f})")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs (no improvement for {patience} epochs)")
            break

# Load best model
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print("\nBest model loaded.")

print("-" * 70)
print("Training completed.")

In [ ]:
# Plot Training Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Plot Loss
axes[0].plot(train_losses, label='Training Loss', marker='o', linewidth=2)
axes[0].plot(val_losses, label='Validation Loss', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training vs Validation Loss', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot Accuracy
axes[1].plot(train_accuracies, label='Training Accuracy', marker='o', linewidth=2)
axes[1].plot(val_accuracies, label='Validation Accuracy', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Training vs Validation Accuracy', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()

# Save curves to working directory (especially useful for Kaggle background runs)
output_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
curves_save_path = os.path.join(output_dir, 'training_curves.png')
plt.savefig(curves_save_path, dpi=300, bbox_inches='tight')
print(f"Training curves saved to: {curves_save_path}")

plt.show()

print(f"\nTraining Summary:")
print(f"  Best Training Accuracy: {max(train_accuracies):.2f}%")
print(f"  Best Validation Accuracy: {max(val_accuracies):.2f}%")
print(f"  Final Validation Accuracy: {val_accuracies[-1]:.2f}%")

In [ ]:
# Evaluate on Test Set
print("Evaluating on test set...")
model.eval()
test_predictions = []
test_true_labels = []
test_correct = 0
test_total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        # Preprocess
        images = (images - MNIST_MEAN) / MNIST_STD
        
        # Forward pass
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        # Track
        test_predictions.extend(predicted.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_accuracy = 100.0 * test_correct / test_total

# Compute metrics
precision = precision_score(test_true_labels, test_predictions, average='weighted')
recall = recall_score(test_true_labels, test_predictions, average='weighted')
f1 = f1_score(test_true_labels, test_predictions, average='weighted')

print(f"\nTest Set Metrics:")
print(f"  Accuracy:  {test_accuracy:.2f}%")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")

# Per-class metrics
print(f"\nPer-Class Accuracy:")
for digit in range(10):
    mask = np.array(test_true_labels) == digit
    if mask.sum() > 0:
        class_acc = np.array(test_predictions)[mask] == digit
        class_acc = 100.0 * class_acc.sum() / mask.sum()
        print(f"  Digit {digit}: {class_acc:.2f}%")

In [ ]:
# Visualize Predictions & Confusion Matrix
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Sample correct predictions
test_loader_viz = DataLoader(test_dataset, batch_size=1, shuffle=False)
correct_count = 0
ax = axes[0, 0]
ax.set_title('Sample Correct Predictions', fontsize=12, fontweight='bold')

with torch.no_grad():
    for images, labels in test_loader_viz:
        if correct_count >= 8:
            break
        images, labels = images.to(device), labels.to(device)
        images_norm = (images - MNIST_MEAN) / MNIST_STD
        outputs = model(images_norm)
        _, predicted = torch.max(outputs, 1)
        
        if predicted == labels:
            ax.text(correct_count * 0.125, 1.05, f"✓ {labels.item()}", 
                   ha='center', va='bottom', fontsize=10, color='green', fontweight='bold')
            correct_count += 1

# Sample incorrect predictions
ax = axes[0, 1]
ax.set_title('Sample Incorrect Predictions', fontsize=12, fontweight='bold')
incorrect_count = 0

with torch.no_grad():
    for images, labels in test_loader_viz:
        if incorrect_count >= 8:
            break
        images, labels = images.to(device), labels.to(device)
        images_norm = (images - MNIST_MEAN) / MNIST_STD
        outputs = model(images_norm)
        _, predicted = torch.max(outputs, 1)
        
        if predicted != labels:
            ax.text(incorrect_count * 0.125, 1.05, 
                   f"✗ {labels.item()}→{predicted.item()}", 
                   ha='center', va='bottom', fontsize=9, color='red', fontweight='bold')
            incorrect_count += 1

# Confusion Matrix
cm = confusion_matrix(test_true_labels, test_predictions)
ax = axes[1, 0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Confusion Matrix (Test Set)', fontsize=12, fontweight='bold')

# Normalized Confusion Matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
ax = axes[1, 1]
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='RdYlGn', ax=ax, 
            cbar_kws={'label': 'Accuracy'}, vmin=0, vmax=1)
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Normalized Confusion Matrix (Accuracy per Class)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Error Analysis: Visualize Misclassified Examples
print("Error Analysis: Finding Misclassified Examples...")

misclassified_indices = []
with torch.no_grad():
    idx = 0
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        images_norm = (images - MNIST_MEAN) / MNIST_STD
        outputs = model(images_norm)
        _, predicted = torch.max(outputs, 1)
        
        for i in range(len(labels)):
            if predicted[i] != labels[i]:
                misclassified_indices.append({
                    'image': images[i].cpu().squeeze().numpy(),
                    'true_label': labels[i].item(),
                    'predicted_label': predicted[i].item(),
                    'confidence': torch.softmax(outputs[i], dim=0).max().item()
                })
        
        if len(misclassified_indices) >= 20:
            break

# Visualize misclassified examples
fig, axes = plt.subplots(4, 5, figsize=(14, 10))
fig.suptitle('Misclassified Examples (Top 20)', fontsize=14, fontweight='bold')

for idx, (ax, sample) in enumerate(zip(axes.flat, misclassified_indices)):
    ax.imshow(sample['image'], cmap='gray')
    true = sample['true_label']
    pred = sample['predicted_label']
    conf = sample['confidence']
    ax.set_title(f"True: {true} → Pred: {pred}\n(conf: {conf:.2f})", 
                color='red', fontsize=9, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print(f"Total misclassified samples: {len(misclassified_indices)}")
print(f"\nMost common misclassifications:")
# Analyze patterns
mistakes = {}
for sample in misclassified_indices:
    pair = (sample['true_label'], sample['predicted_label'])
    mistakes[pair] = mistakes.get(pair, 0) + 1

sorted_mistakes = sorted(mistakes.items(), key=lambda x: x[1], reverse=True)
for (true, pred), count in sorted_mistakes[:10]:
    print(f"  {true} → {pred}: {count} times")

In [ ]:
# Save Model
output_dir = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'

model_save_path = os.path.join(output_dir, 'mnist_ffnn_model.pth')
torch.save(model.state_dict(), model_save_path)
print(f"Model saved to: {model_save_path}")

# Also save a checkpoint with additional info
checkpoint = {
    'model_state_dict': model.state_dict(),
    'test_accuracy': test_accuracy,
    'epoch': len(train_losses),
    'hyperparameters': {
        'input_size': 784,
        'hidden_layers': [128, 64, 32],
        'num_classes': 10,
        'dropout_rate': 0.2,
        'learning_rate': 0.001,
        'batch_size': batch_size
    }
}

checkpoint_path = os.path.join(output_dir, 'mnist_ffnn_checkpoint.pt')
torch.save(checkpoint, checkpoint_path)
print(f"Checkpoint saved to: {checkpoint_path}")

print("\n" + "=" * 70)
print("TRAINING COMPLETE!")
print("=" * 70)
print(f"Final Test Accuracy: {test_accuracy:.2f}%")
print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Total Epochs Trained: {len(train_losses)}")
print("=" * 70)